# Investment AI
## Notebook 05 — Prepare Semantic Search Chunks

The research documents are now parsed and governed.

This notebook uses Databricks `ai_prep_search` to transform those document representations into **semantic, search-ready chunks**.

```text
research.parsed_documents
        ↓
   ai_prep_search
        ↓
 semantic chunks
        ↓
research.document_chunks
        ↓
   NEXT: AI Search
```

This notebook does **not** create an AI Search index yet.  
It prepares the Delta table that will become the index source.


## The key idea: retrieval text vs embedding text

`ai_prep_search` produces two important representations for each chunk:

- **`chunk_to_retrieve`** — the actual source content we want to return as evidence.
- **`chunk_to_embed`** — a context-enriched representation optimized for semantic retrieval.

The embedding representation can include context such as document title, section headers, page references, document context, and table-oriented retrieval hints.

We keep both rather than embedding the raw chunk ourselves.


In [0]:
from pyspark.sql import functions as F

CATALOG = "investment_ai"
RESEARCH_SCHEMA = "research"

PARSED_TABLE = (
    f"{CATALOG}.{RESEARCH_SCHEMA}.parsed_documents"
)

CHUNKS_TABLE = (
    f"{CATALOG}.{RESEARCH_SCHEMA}.document_chunks"
)

print("Parsed source :", PARSED_TABLE)
print("Chunk target  :", CHUNKS_TABLE)


## 1. Verify `ai_prep_search`

`ai_prep_search` accepts either:

- structured `VARIANT` output from `ai_parse_document`, or
- plain text / Markdown `STRING`

That means the PDF/DOCX and HTML/TXT paths from Notebook 04 can now converge into the same retrieval-preparation step.


In [0]:
smoke_test_df = spark.sql("""
SELECT ai_prep_search(
    'Investment AI semantic search preparation test.',
    map('version', '2.0')
) AS result
""")

assert smoke_test_df.first()["result"] is not None

print("✅ ai_prep_search is available.")


## 2. Prepare all 45 documents for semantic retrieval

For PDF and DOCX, we pass the structured `parsed_content`.

For HTML and TXT, we pass `text_content`.

We explicitly use output version `2.0` so the retrieval contract does not silently change with a future default version.


In [0]:
parsed_documents_df = spark.table(PARSED_TABLE)

prepared_documents_df = (
    parsed_documents_df
    .withColumn(
        "search_prep",
        F.when(
            F.col("parsed_content").isNotNull(),
            F.expr(
                """
                ai_prep_search(
                    parsed_content,
                    map('version', '2.0')
                )
                """
            ),
        ).otherwise(
            F.expr(
                """
                ai_prep_search(
                    text_content,
                    map('version', '2.0')
                )
                """
            )
        ),
    )
)

print(
    "Documents prepared for retrieval:",
    prepared_documents_df.count(),
)


In [0]:
prepared_documents_df.display()

## 3. Flatten document-level output into individual chunks

`ai_prep_search` returns one `VARIANT` result per document.

Inside that result is an array of semantic chunks.

We flatten that array so each chunk becomes its own row — exactly the shape we need for a future Delta Sync AI Search index.


In [0]:
prepared_documents_df.createOrReplaceTempView(
    "investment_ai_prepared_documents"
)

chunks_df = spark.sql("""
SELECT
    p.document_id,

    chunk.value:chunk_id::STRING
        AS databricks_chunk_id,

    chunk.value:chunk_position::INT
        AS chunk_position,

    chunk.value:chunk_to_retrieve::STRING
        AS chunk_to_retrieve,

    chunk.value:chunk_to_embed::STRING
        AS chunk_to_embed,

    TO_JSON(chunk.value:pages)
        AS pages_json,

    p.security_id,
    p.company_name,
    p.sector,
    p.fiscal_quarter,
    p.fiscal_year,
    p.document_type,
    p.source_type,
    p.source,
    p.publish_date,
    p.language,
    p.confidentiality,

    p.file_name,
    p.file_extension,
    p.path AS source_path,
    p.source_file_hash,
    p.processing_method

FROM investment_ai_prepared_documents p,
LATERAL variant_explode(
    p.search_prep:document.contents
) AS chunk
""")

print("Semantic chunks generated:", chunks_df.count())


In [0]:
chunks_df.orderBy(F.col("document_id").desc()).display(10)

## 4. Add stable IDs, citation metadata, and source authority

The AI Search index will need a stable primary key.

We create one from:

```text
document_id + chunk_position
```

We also preserve source-authority metadata:

```text
primary   → 1
internal  → 2
secondary → 3
```

This does not automatically force AI Search ranking.  
It gives the Research Agent a governed signal it can use when comparing evidence from different source types.


In [0]:
search_chunks_df = (
    chunks_df
    .withColumn(
        "chunk_id",
        F.concat_ws(
            "::",
            F.col("document_id"),
            F.lpad(
                F.col("chunk_position").cast("string"),
                5,
                "0",
            ),
        ),
    )
    .withColumn(
        "source_authority_rank",
        F.when(
            F.lower(F.col("source_type")) == "primary",
            F.lit(1),
        )
        .when(
            F.lower(F.col("source_type")) == "internal",
            F.lit(2),
        )
        .when(
            F.lower(F.col("source_type")) == "secondary",
            F.lit(3),
        )
        .otherwise(F.lit(9)),
    )
    .withColumn(
        "citation_label",
        F.concat_ws(
            " | ",
            F.col("company_name"),
            F.col("document_type"),
            F.date_format(
                F.col("publish_date"),
                "yyyy-MM-dd",
            ),
            F.col("file_name"),
        ),
    )
    .withColumn(
        "chunk_character_count",
        F.length(F.col("chunk_to_retrieve")),
    )
    .withColumn(
        "prepared_at",
        F.current_timestamp(),
    )
    .select(
        "chunk_id",
        "document_id",
        "databricks_chunk_id",
        "chunk_position",
        "chunk_to_retrieve",
        "chunk_to_embed",
        "chunk_character_count",
        "pages_json",

        "security_id",
        "company_name",
        "sector",
        "fiscal_quarter",
        "fiscal_year",
        "document_type",
        "source_type",
        "source_authority_rank",
        "source",
        "publish_date",
        "language",
        "confidentiality",

        "file_name",
        "file_extension",
        "source_path",
        "source_file_hash",
        "processing_method",
        "citation_label",
        "prepared_at",
    )
)

print("✅ Search-ready chunk dataset created.")


In [0]:
search_chunks_df.display()

## 5. Inspect `chunk_to_retrieve` vs `chunk_to_embed`

This is the most important output to understand in this notebook.

The retrieved text stays close to the source document.

The embedding text is enriched with additional context so semantic search has more information when matching a user's question.


In [0]:
display(
    search_chunks_df
    .select(
        "chunk_id",
        "security_id",
        "document_type",
        "source_type",
        F.substring(
            "chunk_to_retrieve",
            1,
            700,
        ).alias("retrieve_preview"),
        F.substring(
            "chunk_to_embed",
            1,
            1200,
        ).alias("embed_preview"),
    )
    .orderBy(
        "security_id",
        "document_type",
        "chunk_position",
    )
    .limit(10)
)


## 6. Quality checks

These checks protect the future retrieval index from:

- duplicate chunk IDs
- empty retrieval text
- empty embedding text
- missing governed business metadata
- documents that produced no chunks

They are intentionally grouped into one section so the main notebook flow stays focused.


In [0]:
duplicate_chunk_ids = (
    search_chunks_df
    .groupBy("chunk_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_chunks = (
    search_chunks_df
    .filter(
        F.col("chunk_to_retrieve").isNull()
        | (F.length(F.trim(F.col("chunk_to_retrieve"))) == 0)
        | F.col("chunk_to_embed").isNull()
        | (F.length(F.trim(F.col("chunk_to_embed"))) == 0)
        | F.col("security_id").isNull()
        | F.col("document_type").isNull()
        | F.col("source_type").isNull()
    )
    .count()
)

documents_with_chunks = (
    search_chunks_df
    .select("document_id")
    .distinct()
    .count()
)

assert duplicate_chunk_ids == 0
assert invalid_chunks == 0
assert documents_with_chunks == 45

print("✅ Chunk IDs are unique.")
print("✅ Retrieval and embedding text are populated.")
print("✅ Governed metadata is present.")
print("✅ All 45 documents contributed at least one chunk.")


## 7. Publish the AI Search source table

`research.document_chunks` becomes the contract between document engineering and AI Search.

For standard Delta Sync AI Search endpoints, the source Delta table must support Change Data Feed.

We enable CDF now so the table is ready for the next notebook.


In [0]:
(
    search_chunks_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHUNKS_TABLE)
)

spark.sql(f"""
COMMENT ON TABLE {CHUNKS_TABLE} IS
'Search-ready semantic research chunks prepared with Databricks ai_prep_search.'
""")

spark.sql(f"""
ALTER TABLE {CHUNKS_TABLE}
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
)
""")

print(f"✅ Published {CHUNKS_TABLE}")
print("✅ Change Data Feed enabled.")


## 8. Inspect metadata-filtering readiness

Later, retrieval can combine semantic similarity with governed metadata such as:

- security
- quarter and fiscal year
- document type
- source type
- publication date

For example, the Research Agent can narrow a search to primary Q2 2026 evidence for one synthetic company.


In [0]:
display(
    spark.table(CHUNKS_TABLE)
    .filter(
        (F.col("security_id") == "IA-SYN-01")
        & (F.col("fiscal_quarter") == "Q2")
        & (F.col("fiscal_year") == 2026)
        & (F.lower(F.col("source_type")) == "primary")
    )
    .select(
        "chunk_id",
        "document_type",
        "source_type",
        "publish_date",
        "citation_label",
        F.substring(
            "chunk_to_retrieve",
            1,
            600,
        ).alias("chunk_preview"),
    )
    .orderBy(
        "document_type",
        "chunk_position",
    )
)


## 9. Final architecture checkpoint

```text
Raw research documents
        ↓
ai_parse_document / text extraction
        ↓
research.parsed_documents
        ↓
ai_prep_search
        ↓
research.document_chunks
        ↓
Delta + Change Data Feed
        ↓
NEXT: Databricks AI Search
```

The future AI Search index will use:

```text
Primary key       : chunk_id
Embedding source  : chunk_to_embed
Returned evidence : chunk_to_retrieve
Filter metadata   : security_id, company_name, document_type,
                    source_type, fiscal_quarter, fiscal_year,
                    publish_date, source_authority_rank
```



In [0]:
final_chunks_df = spark.table(CHUNKS_TABLE)

summary_df = (
    final_chunks_df
    .groupBy("source_type")
    .agg(
        F.count("*").alias("chunk_count"),
        F.round(
            F.avg("chunk_character_count"),
            1,
        ).alias("avg_chunk_characters"),
    )
    .orderBy("source_type")
)

display(summary_df)

print("Total chunks      :", final_chunks_df.count())
print(
    "Documents covered :",
    final_chunks_df.select("document_id").distinct().count(),
    "/ 45",
)
print("Change Data Feed  : enabled")
print("\nNext: 06_build_ai_search_index.ipynb")
